This notebook is to compare the PGM outputs to the Cernici observations.

In [ ]:
# Import modules
%matplotlib inline

import mikeio
import os
import shutil
import matplotlib.pyplot as plt
from IPython.display import display
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import pathlib
import geopandas

In [ ]:
# Helper functions used across all comparison workflows


# --- LAI / Root Depth helpers ---

def get_valid_data_bbox(da_2d):
    values = np.asarray(da_2d.values)
    valid_mask = np.isfinite(values).any(axis=0)
    if not valid_mask.any():
        return None

    y_idx, x_idx = np.where(valid_mask)
    geom = da_2d.geometry
    x_coords = np.asarray(geom.x)
    y_coords = np.asarray(geom.y)

    return {
        "left": float(x_coords[x_idx].min()),
        "right": float(x_coords[x_idx].max()),
        "bottom": float(y_coords[y_idx].min()),
        "top": float(y_coords[y_idx].max()),
    }


def get_cell_edges(centers):
    centers = np.asarray(centers, dtype=float)
    if centers.ndim != 1 or len(centers) < 2:
        raise ValueError("Coordinate centers must be a 1D array with at least two values.")

    edges = np.empty(len(centers) + 1, dtype=float)
    edges[1:-1] = 0.5 * (centers[:-1] + centers[1:])
    edges[0] = centers[0] - (edges[1] - centers[0])
    edges[-1] = centers[-1] + (centers[-1] - edges[-2])
    return edges


def get_axis_overlap_weights(src_centers, tgt_centers):
    src_edges = get_cell_edges(src_centers)
    tgt_edges = get_cell_edges(tgt_centers)

    n_tgt = len(tgt_centers)
    n_src = len(src_centers)
    weights = np.zeros((n_tgt, n_src), dtype=float)

    for i in range(n_tgt):
        t0 = tgt_edges[i]
        t1 = tgt_edges[i + 1]
        tgt_width = t1 - t0
        if tgt_width <= 0:
            continue

        overlap = np.minimum(t1, src_edges[1:]) - np.maximum(t0, src_edges[:-1])
        overlap = np.clip(overlap, 0.0, None)
        weights[i, :] = overlap / tgt_width

    return weights


def get_time_resolution_category(time_index):
    """Determine if time resolution is hourly, daily, or other."""
    if len(time_index) < 2:
        return "unknown"

    time_diffs = np.diff(time_index.values)
    median_diff = np.median(time_diffs)

    # Convert to timedelta in hours
    median_hours = median_diff / np.timedelta64(1, 'h')

    if median_hours < 2:  # Less than 2 hours → hourly
        return "hourly"
    elif 20 < median_hours < 28:  # Around 24 hours → daily
        return "daily"
    else:
        return f"other ({median_hours:.1f} hours)"


# --- General model-item lookup helpers ---

def find_model_item_by_substring(res_name, item_names, filepath_name):
    """Find a single model item whose name contains ' res_name' (with a leading space).
    This avoids false matches where res_name appears inside a longer token.
    Raises ValueError if no match or multiple matches are found.
    """
    search_str = f" {res_name}"
    matched = [name for name in item_names if search_str in name]
    if not matched:
        raise ValueError(
            f"Item '{res_name}' (searching for '{search_str}') not found in {filepath_name}. Available items: {item_names}"
        )
    if len(matched) > 1:
        raise ValueError(
            f"res_name '{res_name}' matched multiple items in {filepath_name} - "
            f"please update res_name to be more specific. Matching items:\n" +
            "\n".join(f"  - {m}" for m in matched)
        )
    return matched[0]


# --- Porewater helpers ---

def match_model_item_name_by_suffix(target_name, available_names, filename=None):
    file_ref = filename or "<results file>"
    # Match on a suffix with a leading space, e.g. " NH4" or " S_mNH4".
    suffix = f" {target_name.strip()}".lower()
    matches = [
        name for name in available_names
        if name.lower().rstrip().endswith(suffix)
    ]

    if not matches:
        raise ValueError(
            f"Required model item ending with '{suffix}' not found in {file_ref}. "
            f"Available items: {available_names}"
        )

    if len(matches) > 1:
        concentration_matches = [
            name for name in matches
            if "concentration" in name.lower()
        ]

        if len(concentration_matches) == 1:
            return concentration_matches[0]

        if len(concentration_matches) > 1:
            raise ValueError(
                f"Model item suffix '{suffix}' matched multiple 'concentration' items in {file_ref}: "
                f"{concentration_matches}"
            )

        raise ValueError(
            f"Model item suffix '{suffix}' matched multiple items in {file_ref}: {matches}. "
            "None contained 'concentration'."
        )

    return matches[0]


def get_cell_index_for_coordinate(centers, coordinate):
    centers = np.asarray(centers, dtype=float)
    if centers.ndim != 1 or len(centers) < 2:
        raise ValueError("Grid centers must be 1D and contain at least two cells.")

    # Work on sorted centers so descending/unsorted axes are handled correctly.
    order = np.argsort(centers)
    centers_sorted = centers[order]

    edges = np.empty(len(centers_sorted) + 1, dtype=float)
    edges[1:-1] = 0.5 * (centers_sorted[:-1] + centers_sorted[1:])
    edges[0] = centers_sorted[0] - (edges[1] - centers_sorted[0])
    edges[-1] = centers_sorted[-1] + (centers_sorted[-1] - edges[-2])

    idx_sorted = int(np.searchsorted(edges, coordinate, side="right") - 1)
    if idx_sorted < 0 or idx_sorted >= len(centers_sorted):
        raise ValueError(
            f"Coordinate {coordinate} is outside model grid domain "
            f"[{centers_sorted.min()}, {centers_sorted.max()}]."
        )

    return int(order[idx_sorted])


def load_porewater_observations(obs_dir_pw):
    """Load porewater dfs0 observation files, organised by location (C2/C3).
    Returns a dict {loc: {file_stem: {"path", "dataframe", "columns"}}}.
    """
    porewater_obs = {"C2": {}, "C3": {}} #Email from Vaclav on 5-5-2026 confirms that we only use locations C2 and C3
    for obs_file in sorted(obs_dir_pw.glob("*.dfs0")):
        stem_lower = obs_file.stem.lower()
        if "c2" in stem_lower:
            loc_key = "C2"
        elif "c3" in stem_lower:
            loc_key = "C3"
        else:
            continue
        obs_df = mikeio.read(obs_file).to_dataframe()
        porewater_obs[loc_key][obs_file.stem] = {
            "path": obs_file,
            "dataframe": obs_df,
            "columns": list(obs_df.columns),
        }
    return porewater_obs


def load_and_reproject_sampling_locations(obs_dir_pw, model_projection):
    """Load C2/C3 sampling points from shapefile; reproject to model CRS if needed.
    Returns a dict {loc: {"x": float, "y": float}}.
    """
    import geopandas as gpd
    from pyproj import CRS

    locations_dir = obs_dir_pw / "Locations"
    if not locations_dir.exists():
        raise FileNotFoundError(f"Locations folder not found: {locations_dir}")

    shp_files = sorted(locations_dir.glob("*.shp"))
    if not shp_files:
        raise FileNotFoundError(f"No shapefile (*.shp) found in: {locations_dir}")
    if len(shp_files) > 1:
        raise ValueError(f"Multiple shapefiles found in {locations_dir}: {shp_files}. Keep one.")

    locations_shp = shp_files[0]
    locations_gdf = gpd.read_file(locations_shp)

    if "Name" in locations_gdf.columns:
        name_col = "Name"
    elif "Names" in locations_gdf.columns:
        name_col = "Names"
    else:
        raise ValueError(
            f"Column 'Name' not found in {locations_shp}. "
            f"Available columns: {list(locations_gdf.columns)}"
        )

    needed_locations = ["C2", "C3"]
    locations_sel = locations_gdf[
        locations_gdf[name_col].astype(str).str.upper().isin(needed_locations)
    ].copy()
    if locations_sel.empty:
        raise ValueError(f"No C2/C3 points found in {locations_shp} column '{name_col}'.")

    missing = [n for n in needed_locations if n not in set(locations_sel[name_col].astype(str).str.upper())]
    if missing:
        raise ValueError(f"Missing sampling points in shapefile: {missing}")

    if locations_sel.crs is None:
        raise ValueError(f"Shapefile {locations_shp} has no CRS.")
    if model_projection is None or not str(model_projection).strip():
        raise ValueError("Model results projection is missing.")

    locations_crs = CRS.from_user_input(locations_sel.crs)
    try:
        model_crs = CRS.from_user_input(model_projection)
    except Exception as exc:
        raise ValueError(
            f"Could not parse model projection for CRS conversion. "
            f"Model projection string: {model_projection}"
        ) from exc

    if not locations_crs.equals(model_crs):
        print("Projection mismatch detected, converting Locations points to model projection...")
        print(f"  Locations CRS (before): {locations_crs.to_string()}")
        print(f"  Model CRS: {model_crs.to_string()}")
        locations_sel = locations_sel.to_crs(model_crs)
        print(f"  Projection converted to: {model_crs.to_string()}")
    else:
        print(f"Projection check passed without conversion: {locations_crs.to_string()}")

    sampling_points = {}
    for _, row in locations_sel.iterrows():
        loc_name = str(row[name_col]).upper()
        if row.geometry is None or row.geometry.is_empty:
            raise ValueError(f"Location {loc_name} has empty geometry in {locations_shp}")
        sampling_points[loc_name] = {"x": float(row.geometry.x), "y": float(row.geometry.y)}

    print(f"Loaded sampling locations: {locations_shp.name} (column '{name_col}')")
    return sampling_points


def extract_porewater_model_by_location(porewater_res_ds, matched_model_items, model_items_needed, sampling_points, needed_locations=None):
    """Extract model time series at each C2/C3 sampling location.
    Returns a dict {loc: {"x", "y", "x_idx", "y_idx", "items"}}.
    """
    if needed_locations is None:
        needed_locations = ["C2", "C3"]

    first_item_name = matched_model_items[model_items_needed[0]]
    geom = porewater_res_ds[first_item_name].geometry

    porewater_model_by_location = {}
    for loc in needed_locations:
        loc_x = sampling_points[loc]["x"]
        loc_y = sampling_points[loc]["y"]

        x_idx = get_cell_index_for_coordinate(geom.x, loc_x)
        y_idx = get_cell_index_for_coordinate(geom.y, loc_y)

        location_items = {}
        for target_name in model_items_needed:
            actual_name = matched_model_items[target_name]
            da = porewater_res_ds[actual_name]
            # mikeio does not support selecting multiple dimensions in one isel call.
            location_items[target_name] = da.isel(y=y_idx).isel(x=x_idx)

        porewater_model_by_location[loc] = {
            "x": loc_x,
            "y": loc_y,
            "x_idx": x_idx,
            "y_idx": y_idx,
            "items": location_items,
        }

    return porewater_model_by_location


def _normalize_method_label(method_text):
    m = str(method_text).strip().lower().replace("_", " ")
    if "suction" in m:
        return "suctioncups"
    if "lysimeter" in m:
        return "lysimeter"
    return m.replace(" ", "")


def _species_label(species_item, compare_key_text=""):
    text = f"{species_item} {compare_key_text}".lower()
    if "nh4" in text:
        return "NH4"
    if "no3" in text:
        return "NO3"
    return str(species_item).upper()


def _series_from_dataarray(da):
    values = np.asarray(da.values, dtype=float)
    if values.ndim != 1:
        raise ValueError(f"Expected 1D time series after selection, got shape {values.shape} for '{da.name}'")
    return pd.Series(values, index=pd.DatetimeIndex(da.time), name=da.name)


def _column_concentration_series(location_items, species_item):
    numerator_da = location_items[species_item]
    denominator_da = location_items["S_PWV"]

    num_vals = np.asarray(numerator_da.values, dtype=float)
    den_vals = np.asarray(denominator_da.values, dtype=float)

    # If z remains, aggregate to water column first.
    if num_vals.ndim == 2:
        num_vals = np.nansum(num_vals, axis=1)
    if den_vals.ndim == 2:
        den_vals = np.nansum(den_vals, axis=1)

    if num_vals.ndim != 1 or den_vals.ndim != 1:
        raise ValueError(
            f"Column concentration expects 1D or 2D (time,z) arrays, got {num_vals.shape} and {den_vals.shape}"
        )

    finite_den = np.isfinite(den_vals)
    spwv_all_zero = bool(finite_den.any() and np.all(np.abs(den_vals[finite_den]) <= 1.0e-12))

    valid = np.isfinite(num_vals) & finite_den & (np.abs(den_vals) > 1.0e-12)
    conc_vals = np.full_like(num_vals, np.nan, dtype=float)
    conc_vals[valid] = num_vals[valid] / den_vals[valid]

    # g/m3 water equals mg/L numerically.
    series = pd.Series(conc_vals, index=pd.DatetimeIndex(numerator_da.time), name=f"{species_item}_conc_mgL")
    return series, spwv_all_zero


def _layer_series(location_items, species_item, layer_idx):
    da = location_items[species_item]
    vals = np.asarray(da.values, dtype=float)

    if vals.ndim == 1:
        return _series_from_dataarray(da)

    if vals.ndim != 2:
        raise ValueError(f"Layer selection expects 1D or 2D (time,z), got shape {vals.shape} for '{species_item}'")

    if layer_idx < 0 or layer_idx >= vals.shape[1]:
        raise ValueError(
            f"Layer index {layer_idx} out of range for '{species_item}' with {vals.shape[1]} layers"
        )

    return pd.Series(vals[:, layer_idx], index=pd.DatetimeIndex(da.time), name=f"{species_item}_layer{layer_idx}_mgL")


def _species_aliases(species_item):
    s = str(species_item).lower()
    if "nh4" in s:
        return ["nh4", "ammonium"]
    if "no3" in s:
        return ["no3", "nitrate", "nitrates"]
    return [s]


def _find_observation_series_for_case(location_obs_dict, species_item, method_label):
    species_tokens = _species_aliases(species_item)
    method_token = _normalize_method_label(method_label)

    candidates = []
    for file_stem, file_info in location_obs_dict.items():
        df = file_info["dataframe"]
        stem_lower = str(file_stem).lower().replace("_", " ")
        stem_compact = stem_lower.replace(" ", "")

        for col in df.columns:
            col_lower = str(col).lower()
            col_compact = col_lower.replace(" ", "")

            # Require species match first (including aliases, e.g., ammonium/nitrates).
            species_hit_col = any(tok in col_lower for tok in species_tokens)
            species_hit_stem = any(tok in stem_lower for tok in species_tokens)
            if not species_hit_col and not species_hit_stem:
                continue

            score = 0
            if species_hit_col:
                score += 5
            if species_hit_stem:
                score += 1

            if method_token in col_compact:
                score += 3
            if method_token in stem_compact:
                score += 2

            if "concentration" in col_lower:
                score += 1

            candidates.append((score, file_stem, col, df[col]))

    if not candidates:
        return None, None, None

    candidates.sort(key=lambda x: x[0], reverse=True)
    best_score, best_file, best_col, best_series = candidates[0]

    if len(candidates) > 1 and candidates[1][0] == best_score:
        print(
            f"Warning: multiple observation candidates tied for {species_item}/{method_label}. "
            f"Using '{best_file}' -> '{best_col}'."
        )

    return best_series, best_file, best_col


def _safe_name(text):
    return str(text).replace(" ", "_").replace("/", "-").replace("\\", "-")


In [ ]:
# Processing functions for each observation type

def run_harvest_comparison(res_ds, res_item_names, Res_filepath, Output_dir, Obs_dir, crops2compare):
    """Run Harvest dry grain comparison."""
    obs_category = 'Harvest_dry_grain\\BCc'

    for crop, (res_name, obs_name) in crops2compare.items():
        print(f"Comparing {crop} - Result: {res_name} with Observations: {obs_name}")

        units = "g/m²"

        # --- Observations ---
        obs_file = Obs_dir / obs_category / (obs_name + ".dfs0")
        obs_ts = mikeio.read(obs_file).to_dataframe()
        obs_ts.index = obs_ts.index.year
        obs_ts.index.name = "Year"

        # --- Model results ---
        full_item_name = find_model_item_by_substring(res_name, res_item_names, Res_filepath.name)
        da = res_ds[full_item_name]             # DataArray shape: (time, z, y, x)
        top_layer = da.isel(z=-1)           # top layer = highest z index, shape: (time, y, x)

        # Flatten spatial dims to a single "cell" axis
        vals = top_layer.values             # numpy array (time, y, x)
        n_time, n_y, n_x = vals.shape
        vals_2d = vals.reshape(n_time, n_y * n_x)   # (time, n_cells)

        years = pd.Index(da.time.year, name="Year")
        res_df = pd.DataFrame(vals_2d, index=years)

        # Annual max per cell
        res_annual = res_df.groupby(res_df.index).max()

        # Drop cells that are all zero or all NaN across all years
        keep = (res_annual != 0).any(axis=0) & res_annual.notna().any(axis=0)
        res_annual = res_annual.loc[:, keep]

        # Add yearly summary stats across cells
        stats_source = res_annual.copy()
        res_annual["mean"] = stats_source.mean(axis=1)
        res_annual["max"] = stats_source.max(axis=1)
        res_annual["min"] = stats_source.min(axis=1)
        res_annual["median"] = stats_source.median(axis=1)
        res_annual["std"] = stats_source.std(axis=1)

        # --- Compare observations vs model mean (+/- std), keeping all years ---
        obs_series = obs_ts.iloc[:, 0].rename("obs")
        compare_df = pd.concat(
            [obs_series, res_annual[["mean", "std"]].rename(columns={"mean": "model_mean", "std": "model_std"})],
            axis=1,
            join="outer",
        ).sort_index()

        if compare_df["obs"].isna().all() and compare_df["model_mean"].isna().all():
            print("No values available for plotting.")
            continue

        # Plot all years; missing values become zero-height bars
        obs_plot = compare_df["obs"].fillna(0.0)
        model_plot = compare_df["model_mean"].fillna(0.0)
        model_err = compare_df["model_std"].fillna(0.0)

        x = np.arange(len(compare_df.index))
        width = 0.4
        obs_color = "tab:blue"
        res_color = "tab:orange"

        fig, ax = plt.subplots(figsize=(12, 6))
        ax.bar(x - width / 2, obs_plot, width, label="Observation", color=obs_color)
        ax.bar(
            x + width / 2,
            model_plot,
            width,
            yerr=model_err,
            capsize=4,
            color=res_color,
            error_kw={"ecolor": res_color},
            label="Model mean +/- std",
        )

        ax.set_xticks(x)
        ax.set_xticklabels(compare_df.index.astype(str), rotation=45)
        ax.set_xlabel("Year")
        ax.set_ylabel(f"{crop}, ({units})")
        ax.set_title(f"{crop}: Observations vs Model ({obs_name} vs {res_name})")
        ax.legend()
        ax.grid(axis="y", alpha=0.3)
        fig.tight_layout()

        # Save plot and combined annual timeseries for this crop in the obs_category output folder
        output_subdir = Output_dir / obs_category
        output_subdir.mkdir(parents=True, exist_ok=True)

        safe_crop = crop.replace(" ", "_").replace("/", "-").replace("\\", "-")
        safe_obs = obs_name.replace(" ", "_").replace("/", "-").replace("\\", "-")
        safe_res = res_name.replace(" ", "_").replace("/", "-").replace("\\", "-")
        out_stem = f"{safe_crop}_{safe_obs}_vs_{safe_res}"

        png_path = output_subdir / f"{out_stem}.png"
        fig.savefig(png_path, dpi=300, bbox_inches="tight")

        dfs0_df = compare_df[["obs", "model_mean", "model_std"]].copy()
        dfs0_df.index = pd.to_datetime(dfs0_df.index.astype(str) + "-01-01")
        dfs0_df.columns = [
            f"Obs {obs_name}",
            f"Res {res_name} mean",
            f"Res {res_name} std",
        ]

        dfs0_path = output_subdir / f"{out_stem}.dfs0"
        mikeio.from_pandas(dfs0_df).to_dfs(dfs0_path)

        plt.close(fig)


def run_et_comparison(res_ds, res_item_names, Res_filepath, Output_dir, Obs_dir, ET2compare):
    """Run Evapotranspiration comparison."""
    obs_category = 'ET'

    for et_type, (res_name, obs_name) in ET2compare.items():
        print(f"Comparing {et_type} - Result: {res_name} with Observations: {obs_name}")

        et_units = "mm/d"
        obs_item_by_et_type = {
            "Actual Evapotranspiration": "ETa",
            "Reference Evapotranspiration": "Actual Evap",  # ETr obs file has multiple items
        }

        # --- Observations (daily dfs0; keep original daily timestamps) ---
        obs_file = Obs_dir / obs_category / (obs_name + ".dfs0")
        obs_df = mikeio.read(obs_file).to_dataframe()

        obs_item_name = obs_item_by_et_type.get(et_type)
        if obs_item_name is None:
            obs_series = obs_df.iloc[:, 0]
        elif obs_item_name in obs_df.columns:
            obs_series = obs_df[obs_item_name]
        else:
            obs_matches = [col for col in obs_df.columns if obs_item_name in col]
            if not obs_matches:
                raise ValueError(
                    f"Observation item '{obs_item_name}' not found in {obs_file.name}. "
                    f"Available items: {list(obs_df.columns)}"
                )
            if len(obs_matches) > 1:
                raise ValueError(
                    f"Observation item '{obs_item_name}' matched multiple items in {obs_file.name}: {obs_matches}"
                )
            obs_series = obs_df[obs_matches[0]]

        obs_series = obs_series.rename("obs")

        # --- Model results ---
        full_item_name = find_model_item_by_substring(res_name, res_item_names, Res_filepath.name)
        da = res_ds[full_item_name]             # DataArray shape: (time, z, y, x)
        top_layer = da.isel(z=-1)           # top layer = highest z index, shape: (time, y, x)

        # Flatten top layer spatial dims to a single "cell" axis
        vals = top_layer.values             # numpy array (time, y, x)
        n_time, n_y, n_x = vals.shape
        vals_2d = vals.reshape(n_time, n_y * n_x)   # (time, n_cells)

        res_df = pd.DataFrame(vals_2d, index=pd.DatetimeIndex(da.time))

        # Drop cells that are all zero or all NaN across all timestamps
        keep = (res_df != 0).any(axis=0) & res_df.notna().any(axis=0)
        res_df = res_df.loc[:, keep]
        if res_df.shape[1] == 0:
            raise ValueError(
                f"No valid top-layer cells found for '{full_item_name}' after filtering all-zero/all-NaN cells."
            )

        # Daily summary stats across selected cells
        stats_source = res_df.copy()
        model_stats = pd.DataFrame(index=stats_source.index)
        model_stats["model_mean"] = stats_source.mean(axis=1)
        model_stats["model_max"] = stats_source.max(axis=1)
        model_stats["model_min"] = stats_source.min(axis=1)
        model_stats["model_median"] = stats_source.median(axis=1)
        model_stats["model_std"] = stats_source.std(axis=1)

        # --- Compare observations vs model stats on daily index ---
        compare_df = pd.concat([obs_series, model_stats], axis=1, join="outer").sort_index()

        if compare_df["obs"].isna().all() and compare_df["model_mean"].isna().all():
            print("No values available for plotting.")
            continue

        # Ensure equidistant daily timestep for dfs0 output
        daily_index = pd.date_range(compare_df.index.min(), compare_df.index.max(), freq="D")
        compare_df = compare_df.reindex(daily_index)

        # Line plot comparison (obs + model mean +/- std)
        fig, ax = plt.subplots(figsize=(12, 6))
        obs_color = "tab:blue"
        res_color = "tab:orange"
        ax.plot(compare_df.index, compare_df["obs"], label=f"Observation ({obs_item_name})", linewidth=1.5, color=obs_color)
        ax.plot(compare_df.index, compare_df["model_mean"], label=f"Model mean ({full_item_name})", linewidth=1.7, color=res_color)

        lower = compare_df["model_mean"] - compare_df["model_std"]
        upper = compare_df["model_mean"] + compare_df["model_std"]
        ax.fill_between(compare_df.index, lower, upper, alpha=0.2, color=res_color, label="Model mean +/- std")

        ax.set_xlabel("Date")
        ax.set_ylabel(f"{et_type}, ({et_units})")
        ax.set_title(f"{et_type}: Observations vs Model ({obs_name} vs {res_name})")
        ax.legend()
        ax.grid(axis="y", alpha=0.3)
        fig.tight_layout()

        # Save outputs in the obs_category output folder
        output_subdir = Output_dir / obs_category
        output_subdir.mkdir(parents=True, exist_ok=True)

        safe_type = et_type.replace(" ", "_").replace("/", "-").replace("\\", "-")
        safe_obs = obs_name.replace(" ", "_").replace("/", "-").replace("\\", "-")
        safe_res = res_name.replace(" ", "_").replace("/", "-").replace("\\", "-")
        out_stem = f"{safe_type}_{safe_obs}_vs_{safe_res}"

        png_path = output_subdir / f"{out_stem}.png"
        fig.savefig(png_path, dpi=300, bbox_inches="tight")

        dfs0_df = compare_df[["obs", "model_mean", "model_std", "model_max", "model_min", "model_median"]].copy()
        dfs0_df.columns = [
            f"Obs {obs_name}",
            f"Res {res_name} mean",
            f"Res {res_name} std",
            f"Res {res_name} max",
            f"Res {res_name} min",
            f"Res {res_name} median",
        ]

        dfs0_path = output_subdir / f"{out_stem}.dfs0"
        mikeio.from_pandas(dfs0_df).to_dfs(dfs0_path)

        plt.close(fig)


def run_lai_rd_comparison(res_ds, res_item_names, Res_filepath, Output_dir, Obs_dir, eo2compare, obs2compare):
    """Run LAI and Root Depth comparison."""
    obs_categories = ['LAI', 'Root Depth']

    for eo_type, (res_name, obs_name) in eo2compare.items():
        if eo_type not in obs2compare:
            continue

        print(f"Comparing {eo_type} - Result: {res_name} with Observations: {obs_name}")

        # --- Observations (daily dfs2; kept as DataArray) ---
        obs_file = Obs_dir / eo_type / (obs_name + ".dfs2")
        obs_ds = mikeio.read(obs_file)

        item_names = [item.name for item in obs_ds.items]
        if eo_type == "LAI":
            name_tokens = ["LAI"]
        else:
            name_tokens = ["RD", "Root Depth"]

        matched_items = [
            name for name in item_names
            if any(token.lower() in name.lower() for token in name_tokens)
        ]

        if not matched_items:
            raise ValueError(
                f"No observation item containing {name_tokens} found in {obs_file.name}. "
                f"Available items: {item_names}"
            )
        if len(matched_items) > 1:
            raise ValueError(
                f"Multiple observation items matched {name_tokens} in {obs_file.name}: {matched_items}"
            )

        obs_item_name = matched_items[0]
        obs_da = obs_ds[obs_item_name]

        # --- Model results (from res_ds; top layer only) ---
        if eo_type == "LAI":
            matched_res_items = [
                name for name in res_item_names
                if name.lower().rstrip().endswith(" lai")
            ]
        else:
            matched_res_items = [
                name for name in res_item_names
                if name.lower().rstrip().endswith(" rd")
            ]

        if not matched_res_items:
            expected_suffix = " LAI" if eo_type == "LAI" else " RD"
            raise ValueError(
                f"No model item ending with '{expected_suffix}' found in {Res_filepath.name}. "
                f"Available items: {res_item_names}"
            )
        if len(matched_res_items) > 1:
            raise ValueError(
                f"Multiple model items matched suffix for {eo_type} in {Res_filepath.name}: {matched_res_items}"
            )

        res_item_name = matched_res_items[0]
        res_da = res_ds[res_item_name]
        res_top_layer = res_da.isel(z=-1)

        # Root Depth unit harmonization: model is in m, observations are in mm.
        if eo_type == "Root Depth":
            res_top_layer = mikeio.DataArray(
                data=np.asarray(res_top_layer.values, dtype=float) * 1000.0,
                time=pd.DatetimeIndex(res_top_layer.time),
                name=res_top_layer.name,
                type=res_top_layer.type,
                unit=obs_da.unit,
                geometry=res_top_layer.geometry,
            )
            print("Converted Root Depth model results from m to mm.")

        # --- Check time resolution match ---
        obs_times_check = pd.DatetimeIndex(obs_da.time)
        model_times_check = pd.DatetimeIndex(res_top_layer.time)
        obs_res_category = get_time_resolution_category(obs_times_check)
        model_res_category = get_time_resolution_category(model_times_check)

        if obs_res_category != model_res_category:
            raise ValueError(
                f"Time resolution mismatch for {eo_type}: "
                f"observations are {obs_res_category}, model results are {model_res_category}. "
                f"Cannot proceed with comparison."
            )

        # --- Location check: projections must match and valid-data footprints must overlap ---
        obs_projection = getattr(obs_da.geometry, "projection_string", getattr(obs_da.geometry, "projection", None))
        res_projection = getattr(res_top_layer.geometry, "projection_string", getattr(res_top_layer.geometry, "projection", None))

        if obs_projection != res_projection:
            print(
                f"Warning: {eo_type} observation and model projections differ; "
                f"stopping processing for this comparison.\n"
                f"  Observation projection: {obs_projection}\n"
                f"  Model projection: {res_projection}"
            )
            continue

        obs_valid_bbox = get_valid_data_bbox(obs_da)
        res_valid_bbox = get_valid_data_bbox(res_top_layer)

        if obs_valid_bbox is None or res_valid_bbox is None:
            print(
                f"Warning: {eo_type} has no valid spatial data in "
                f"{'observations' if obs_valid_bbox is None else 'model results'}; "
                "stopping processing for this comparison."
            )
            continue

        overlap_left = max(obs_valid_bbox["left"], res_valid_bbox["left"])
        overlap_right = min(obs_valid_bbox["right"], res_valid_bbox["right"])
        overlap_bottom = max(obs_valid_bbox["bottom"], res_valid_bbox["bottom"])
        overlap_top = min(obs_valid_bbox["top"], res_valid_bbox["top"])

        model_width = max(0.0, res_valid_bbox["right"] - res_valid_bbox["left"])
        model_height = max(0.0, res_valid_bbox["top"] - res_valid_bbox["bottom"])
        model_area = model_width * model_height

        overlap_width = max(0.0, overlap_right - overlap_left)
        overlap_height = max(0.0, overlap_top - overlap_bottom)
        overlap_area = overlap_width * overlap_height

        model_coverage_pct = 0.0 if model_area <= 0 else (overlap_area / model_area) * 100.0

        has_overlap = (overlap_left <= overlap_right) and (overlap_bottom <= overlap_top)
        if not has_overlap:
            print(
                f"Warning: {eo_type} observation and model data footprints do not overlap; "
                "stopping processing for this comparison.\n"
                f"  Observation bbox: {obs_valid_bbox}\n"
                f"  Model bbox: {res_valid_bbox}\n"
                f"  Observation coverage of model area: {model_coverage_pct:.2f}%"
            )
            continue

        print(
            f"Location check passed for {eo_type}.\n"
            f"  Overlapping bbox: {{'left': {overlap_left}, 'right': {overlap_right}, "
            f"'bottom': {overlap_bottom}, 'top': {overlap_top}}}\n"
            f"  Observation coverage of model area: {model_coverage_pct:.2f}%"
        )

        # --- Area-weighted regridding: observations -> model grid ---
        obs_vals = np.asarray(obs_da.values, dtype=float)  # (time, y_obs, x_obs)
        obs_x = np.asarray(obs_da.geometry.x, dtype=float)
        obs_y = np.asarray(obs_da.geometry.y, dtype=float)
        res_x = np.asarray(res_top_layer.geometry.x, dtype=float)
        res_y = np.asarray(res_top_layer.geometry.y, dtype=float)

        obs_x_order = np.argsort(obs_x)
        obs_y_order = np.argsort(obs_y)
        res_x_order = np.argsort(res_x)
        res_y_order = np.argsort(res_y)

        obs_x_sorted = obs_x[obs_x_order]
        obs_y_sorted = obs_y[obs_y_order]
        res_x_sorted = res_x[res_x_order]
        res_y_sorted = res_y[res_y_order]

        obs_vals_sorted = obs_vals[:, obs_y_order, :][:, :, obs_x_order]

        wx = get_axis_overlap_weights(obs_x_sorted, res_x_sorted)  # (x_model, x_obs)
        wy = get_axis_overlap_weights(obs_y_sorted, res_y_sorted)  # (y_model, y_obs)

        valid_mask = np.isfinite(obs_vals_sorted)
        obs_vals_filled = np.where(valid_mask, obs_vals_sorted, 0.0)
        valid_mask_float = valid_mask.astype(float)

        num_x = np.einsum("ix,tyx->tyi", wx, obs_vals_filled)
        den_x = np.einsum("ix,tyx->tyi", wx, valid_mask_float)

        numerator_sorted = np.einsum("jy,tyi->tji", wy, num_x)
        denominator_sorted = np.einsum("jy,tyi->tji", wy, den_x)

        obs_regridded_sorted = np.where(denominator_sorted > 0, numerator_sorted / denominator_sorted, np.nan)

        inv_res_x_order = np.argsort(res_x_order)
        inv_res_y_order = np.argsort(res_y_order)
        obs_regridded_vals = obs_regridded_sorted[:, inv_res_y_order, :][:, :, inv_res_x_order]

        obs_regridded_to_model = mikeio.DataArray(
            data=obs_regridded_vals,
            time=pd.DatetimeIndex(obs_da.time),
            name=f"obs_{eo_type}_regridded",
            type=obs_da.type,
            unit=obs_da.unit,
            geometry=res_top_layer.geometry,
        )

        print(
            f"Area-weighted regridding completed for {eo_type}. "
            f"Regridded observation shape: {obs_regridded_to_model.values.shape}"
        )

        # --- Time alignment: filter to model time span ---
        model_times = pd.DatetimeIndex(res_top_layer.time)
        obs_times = pd.DatetimeIndex(obs_regridded_to_model.time)
        obs_times_overlap = pd.DatetimeIndex(obs_da.time)

        model_time_min = model_times.min()
        model_time_max = model_times.max()

        obs_in_range_mask = (obs_times >= model_time_min) & (obs_times <= model_time_max)
        obs_in_range_mask_overlap = (obs_times_overlap >= model_time_min) & (obs_times_overlap <= model_time_max)

        obs_regridded_time_filtered = obs_regridded_to_model.isel(time=obs_in_range_mask)
        obs_da_time_filtered = obs_da.isel(time=obs_in_range_mask_overlap)

        # --- Compute comparison metric: (model - obs_regridded) / model ---
        if len(model_times) < 2:
            raise ValueError(f"Need at least 2 model timesteps for alignment in {eo_type}.")

        model_step = pd.to_timedelta(np.median(np.diff(model_times.values)))
        align_tolerance = model_step / 2

        obs_filtered_times = pd.DatetimeIndex(obs_regridded_time_filtered.time)
        time_indexer = obs_filtered_times.get_indexer(model_times, method="nearest", tolerance=align_tolerance)
        if (time_indexer < 0).any():
            raise ValueError(
                f"Could not align observation times to model times within tolerance ({align_tolerance}) for {eo_type}."
            )

        obs_regridded_aligned_vals = obs_regridded_time_filtered.values[time_indexer, :, :]
        obs_regridded_aligned = mikeio.DataArray(
            data=obs_regridded_aligned_vals,
            time=model_times,
            name=obs_regridded_to_model.name,
            type=obs_regridded_to_model.type,
            unit=obs_regridded_to_model.unit,
            geometry=obs_regridded_to_model.geometry,
        )

        res_vals = np.asarray(res_top_layer.values, dtype=float)
        obs_vals_aligned = np.asarray(obs_regridded_aligned.values, dtype=float)

        valid_ratio_mask = np.isfinite(res_vals) & np.isfinite(obs_vals_aligned) & (np.abs(res_vals) > 1.0e-12)
        comparison_vals = np.full_like(res_vals, np.nan, dtype=float)
        comparison_vals[valid_ratio_mask] = (res_vals[valid_ratio_mask] - obs_vals_aligned[valid_ratio_mask]) / res_vals[valid_ratio_mask]

        n_total = comparison_vals.size
        n_valid_res = int(np.isfinite(res_vals).sum())
        n_valid_obs = int(np.isfinite(obs_vals_aligned).sum())
        n_nonzero_res = int((np.isfinite(res_vals) & (np.abs(res_vals) > 1.0e-12)).sum())
        n_valid_ratio = int(np.isfinite(comparison_vals).sum())

        print(
            f"Comparison diagnostics for {eo_type}: valid model cells={n_valid_res}/{n_total}, "
            f"valid regridded obs cells={n_valid_obs}/{n_total}, non-zero model cells={n_nonzero_res}/{n_total}, "
            f"valid ratio cells={n_valid_ratio}/{n_total}"
        )
        if n_valid_ratio == 0:
            print(
                f"Warning: '(res-obs)/res' is fully NaN for {eo_type}. "
                f"Likely causes: no overlapping valid values after regridding/time alignment, or model values are zero."
            )

        comparison_metric = mikeio.DataArray(
            data=comparison_vals,
            time=model_times,
            name="(res-obs)/res",
            type=obs_regridded_to_model.type,
            unit=obs_regridded_to_model.unit,
            geometry=res_top_layer.geometry,
        )

        res_for_export = mikeio.DataArray(
            data=np.asarray(res_top_layer.values, dtype=float),
            time=model_times,
            name=f"res_{eo_type}",
            type=res_top_layer.type,
            unit=res_top_layer.unit,
            geometry=res_top_layer.geometry,
        )

        print(
            f"Time alignment: using {len(model_times)} model timesteps "
            f"from {model_time_min} to {model_time_max}"
        )

        # Export both original and regridded observations for all observation categories
        x_edges_sorted = get_cell_edges(obs_x_sorted)
        y_edges_sorted = get_cell_edges(obs_y_sorted)

        x_overlap_sorted_mask = (
            np.minimum(x_edges_sorted[1:], overlap_right) - np.maximum(x_edges_sorted[:-1], overlap_left)
        ) > 0.0
        y_overlap_sorted_mask = (
            np.minimum(y_edges_sorted[1:], overlap_top) - np.maximum(y_edges_sorted[:-1], overlap_bottom)
        ) > 0.0

        if not x_overlap_sorted_mask.any() or not y_overlap_sorted_mask.any():
            print(
                f"Warning: No observation cells intersect the overlap bbox at cell level; "
                f"skipping {eo_type} export."
            )
            continue

        obs_x_overlap_idx = obs_x_order[x_overlap_sorted_mask]
        obs_y_overlap_idx = obs_y_order[y_overlap_sorted_mask]
        obs_da_overlap = obs_da_time_filtered.isel(y=obs_y_overlap_idx, x=obs_x_overlap_idx)

        eo_output_dir = Output_dir / eo_type
        eo_output_dir.mkdir(parents=True, exist_ok=True)

        obs_original_path = eo_output_dir / f"obs_{eo_type}_original.dfs2"
        obs_regridded_path = eo_output_dir / f"{eo_type}_compare.dfs2"

        obs_da_overlap.to_dataset().to_dfs(obs_original_path)
        regridded_dataset = mikeio.Dataset([obs_regridded_aligned, res_for_export, comparison_metric])
        regridded_dataset.to_dfs(obs_regridded_path)

        print(
            f"Saved overlap-only original {eo_type} observations to {obs_original_path}\n"
            f"Saved regridded {eo_type} observations (+ model results + (res-obs)/res) to {obs_regridded_path}"
        )

        # --- Time-series processing on model-valid cells only ---
        res_values = np.asarray(res_for_export.values, dtype=float)
        obs_values = np.asarray(obs_regridded_aligned.values, dtype=float)

        any_finite_cell_mask = np.isfinite(res_values).any(axis=0)
        any_nonzero_cell_mask = (np.abs(res_values) > 1.0e-12).any(axis=0)
        model_valid_cell_mask = (np.isfinite(res_values) & (np.abs(res_values) > 1.0e-12)).any(axis=0)
        print(
            f"Cell diagnostics for {eo_type}: any finite={int(any_finite_cell_mask.sum())}, "
            f"any non-zero={int(any_nonzero_cell_mask.sum())}, selected={int(model_valid_cell_mask.sum())}"
        )
        n_selected_cells = int(model_valid_cell_mask.sum())
        if n_selected_cells == 0:
            print(
                f"Warning: No model cells have non-zero and non-NaN values at any timestep for {eo_type}; "
                f"skipping time-series statistics export."
            )
            continue

        res_selected = res_values[:, model_valid_cell_mask]
        obs_selected = obs_values[:, model_valid_cell_mask]

        res_selected_df = pd.DataFrame(res_selected, index=model_times)
        obs_selected_df = pd.DataFrame(obs_selected, index=model_times)

        ts_stats = pd.DataFrame(index=model_times)
        ts_stats["obs_mean"] = obs_selected_df.mean(axis=1)
        ts_stats["obs_max"] = obs_selected_df.max(axis=1)
        ts_stats["obs_min"] = obs_selected_df.min(axis=1)
        ts_stats["obs_median"] = obs_selected_df.median(axis=1)
        ts_stats["obs_std"] = obs_selected_df.std(axis=1)
        ts_stats["res_mean"] = res_selected_df.mean(axis=1)
        ts_stats["res_max"] = res_selected_df.max(axis=1)
        ts_stats["res_min"] = res_selected_df.min(axis=1)
        ts_stats["res_median"] = res_selected_df.median(axis=1)
        ts_stats["res_std"] = res_selected_df.std(axis=1)

        fig, ax = plt.subplots(figsize=(12, 6))
        ax.plot(ts_stats.index, ts_stats["obs_mean"], label="Obs mean", linewidth=1.7)
        ax.plot(ts_stats.index, ts_stats["res_mean"], label="Res mean", linewidth=1.7)

        obs_lower = ts_stats["obs_mean"] - ts_stats["obs_std"]
        obs_upper = ts_stats["obs_mean"] + ts_stats["obs_std"]
        res_lower = ts_stats["res_mean"] - ts_stats["res_std"]
        res_upper = ts_stats["res_mean"] + ts_stats["res_std"]

        ax.fill_between(ts_stats.index, obs_lower, obs_upper, alpha=0.2, label="Obs mean +/- std")
        ax.fill_between(ts_stats.index, res_lower, res_upper, alpha=0.2, label="Res mean +/- std")

        ax.set_xlabel("Date")
        ax.set_ylabel(eo_type)
        ax.set_title(f"{eo_type}: Spatial Time-Series Statistics ({n_selected_cells} selected cells)")
        ax.legend()
        ax.grid(axis="y", alpha=0.3)
        fig.tight_layout()

        ts_png_path = eo_output_dir / f"{eo_type}_compare.png"
        fig.savefig(ts_png_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

        ts_dfs0_path = eo_output_dir / f"{eo_type}_compare.dfs0"
        ts_stats_output = ts_stats[["obs_mean", "obs_std", "obs_max", "obs_min", "obs_median", "res_mean", "res_std", "res_max", "res_min", "res_median"]].copy()
        ts_stats_output.columns = [
            f"Obs {eo_type} mean",
            f"Obs {eo_type} std",
            f"Obs {eo_type} max",
            f"Obs {eo_type} min",
            f"Obs {eo_type} median",
            f"Res {eo_type} mean",
            f"Res {eo_type} std",
            f"Res {eo_type} max",
            f"Res {eo_type} min",
            f"Res {eo_type} median",
        ]
        mikeio.from_pandas(ts_stats_output).to_dfs(ts_dfs0_path)

        print(
            f"Saved {eo_type} time-series plot to {ts_png_path}\n"
            f"Saved {eo_type} time-series statistics to {ts_dfs0_path}"
        )


def run_porewater_comparison(res_ds, res_item_names, Res_filepath, Output_dir, Obs_dir, porewater2compare, lysimeter_layer, suctioncup_layer):
    """Run Porewater and Groundwater N/P comparison."""
    obs_category = "Soil_water_N_P"
    porewater_output_dir = Output_dir / obs_category
    porewater_output_dir.mkdir(parents=True, exist_ok=True)

    obs_dir_pw = Obs_dir / obs_category
    if not obs_dir_pw.exists():
        raise FileNotFoundError(f"Observation folder not found: {obs_dir_pw}")

    # --- Load observations and identify required model items ---
    porewater_obs = load_porewater_observations(obs_dir_pw)

    compare_model_items = sorted({mapping[0] for mapping in porewater2compare.values()})
    model_items_needed = sorted(set(compare_model_items + ["S_PWV"]))

    matched_model_items = {
        target: match_model_item_name_by_suffix(target, res_item_names, Res_filepath.name)
        for target in model_items_needed
    }

    # --- Load sampling locations and extract model data at each point ---
    first_da = res_ds[matched_model_items[model_items_needed[0]]]
    model_projection = getattr(first_da.geometry, "projection_string", getattr(first_da.geometry, "projection", None))

    sampling_points = load_and_reproject_sampling_locations(obs_dir_pw, model_projection)
    porewater_model_by_location = extract_porewater_model_by_location(
        res_ds, matched_model_items, model_items_needed, sampling_points
    )

    # --- Diagnostics ---
    print(f"\nLoaded pore-water observations from: {obs_dir_pw}")
    for loc in ["C2", "C3"]: #Email from Vaclav on 5-5-2026 confirms that we only use locations C2 and C3
        print(f"  {loc}: {len(porewater_obs[loc])} files")
    print(f"Model results file: {Res_filepath.name}")
    print(f"Selected model items: {model_items_needed}")
    for target_name, actual_name in matched_model_items.items():
        print(f"  {target_name} -> {actual_name}")
    for loc in ["C2", "C3"]: #Email from Vaclav on 5-5-2026 confirms that we only use locations C2 and C3
        info = porewater_model_by_location[loc]
        print(f"  {loc}: grid cell (x_idx={info['x_idx']}, y_idx={info['y_idx']}) at x={info['x']:.1f}, y={info['y']:.1f}")

    # --- Comparison loop ---
    n_comparisons = 0
    combined_obs_by_location = {loc: {} for loc in ["C2", "C3"]} #Email from Vaclav on 5-5-2026 confirms that we only use locations C2 and C3
    combined_model_by_location = {loc: {} for loc in ["C2", "C3"]}

    for compare_key, (model_base_item, obs_method) in porewater2compare.items():
        for loc in ["C2", "C3"]:
            location_items = porewater_model_by_location[loc]["items"]
            if model_base_item not in location_items:
                print(f"Skipping {compare_key} / {loc}: model item '{model_base_item}' not extracted.")
                continue

            key_lower = compare_key.lower()
            zero_spwv_warning = False
            if "column" in key_lower:
                model_series, spwv_all_zero = _column_concentration_series(location_items, model_base_item)
                zero_spwv_warning = model_series.isna().all() and spwv_all_zero
            elif "layer" in key_lower:
                if "porewater" in key_lower:
                    layer_idx = suctioncup_layer
                elif "groundwater" in key_lower:
                    layer_idx = lysimeter_layer
                else:
                    print(f"Skipping {compare_key} / {loc}: could not infer layer rule from key.")
                    continue
                model_series = _layer_series(location_items, model_base_item, layer_idx)
            else:
                print(f"Skipping {compare_key} / {loc}: key must contain 'Column' or 'Layer'.")
                continue

            obs_series, obs_file, obs_col = _find_observation_series_for_case(
                porewater_obs[loc], model_base_item, obs_method,
            )
            if obs_series is None:
                print(f"Skipping {compare_key} / {loc}: no matching observation series found.")
                continue

            compare_df = pd.concat(
                [obs_series.rename("obs"), model_series.rename("model")],
                axis=1, join="outer", sort=True,
            ).sort_index()

            if zero_spwv_warning:
                print(
                    f"Warning: {compare_key} / {loc}: model column concentration for '{model_base_item}' is all NaN "
                    "because S_PWV is zero at all finite timesteps in this cell. Plotting anyway."
                )

            if compare_df["obs"].isna().all() and compare_df["model"].isna().all() and not zero_spwv_warning:
                print(f"Skipping {compare_key} / {loc}: both series are empty.")
                continue

            method_norm = _normalize_method_label(obs_method)
            species_name = _species_label(model_base_item, compare_key)
            obs_series_name = f"obs_{method_norm}_{species_name}"
            model_col_name = f"res_{_safe_name(compare_key)}"

            new_obs_series = compare_df["obs"].rename(obs_series_name)
            if obs_series_name in combined_obs_by_location[loc]:
                combined_obs_by_location[loc][obs_series_name] = combined_obs_by_location[loc][obs_series_name].combine_first(new_obs_series)
            else:
                combined_obs_by_location[loc][obs_series_name] = new_obs_series

            combined_model_by_location[loc][model_col_name] = compare_df["model"].rename(model_col_name)

            fig, ax = plt.subplots(figsize=(12, 6))
            ax.scatter(compare_df.index, compare_df["obs"], label=f"obs {method_norm} {species_name} mgL", s=18, color="tab:blue", zorder=3)
            ax.plot(compare_df.index, compare_df["model"], label=f"res {compare_key} mgL", linewidth=1.6, color="tab:orange")
            ax.set_xlabel("Date")
            ax.set_ylabel("Concentration (mg/L)")
            ax.set_title(f"{compare_key} {loc}")
            ax.legend()
            ax.grid(axis="y", alpha=0.3)
            fig.tight_layout()

            png_path = porewater_output_dir / f"{_safe_name(f'{compare_key}_{loc}')}.png"
            fig.savefig(png_path, dpi=300, bbox_inches="tight")
            plt.close(fig)

            n_comparisons += 1
            print(
                f"Saved {png_path} using observation '{obs_series_name}' from source '{obs_file}' "
                f"column '{obs_col}' and model item '{model_base_item}' at {loc}."
            )

    print(f"\nCompleted pore-water comparison plots: {n_comparisons}")

    # --- Combined per-location output (all obs + model series in one dfs0 + plot) ---
    for loc in ["C2", "C3"]: #Email from Vaclav on 5-5-2026 confirms that we only use locations C2 and C3
        obs_dict = combined_obs_by_location[loc]
        model_dict = combined_model_by_location[loc]

        if not obs_dict and not model_dict:
            print(f"Skipping combined output for {loc}: no pore-water series available.")
            continue

        obs_cols = sorted(obs_dict.keys())
        model_cols = sorted(model_dict.keys())
        print(f"{loc} observation series included: {obs_cols}")

        ordered_series = [obs_dict[n].rename(n) for n in obs_cols] + [model_dict[n].rename(n) for n in model_cols]
        combined_df = pd.concat(ordered_series, axis=1, join="outer", sort=True).sort_index()

        combined_dfs0_path = porewater_output_dir / f"Porewater_AllSeries_{loc}.dfs0"
        mikeio.from_pandas(combined_df).to_dfs(combined_dfs0_path)

        fig, ax = plt.subplots(figsize=(14, 8))
        for obs_col_name in obs_cols:
            ax.scatter(combined_df.index, combined_df[obs_col_name], label=obs_col_name, s=12, alpha=0.85, zorder=3)
        for model_col_name in model_cols:
            ax.plot(combined_df.index, combined_df[model_col_name], label=model_col_name, linewidth=1.3, alpha=0.9)
        ax.set_xlabel("Date")
        ax.set_ylabel("Concentration (mg/L)")
        ax.set_title(f"All pore-water observation/model series at {loc}")
        ax.grid(axis="y", alpha=0.3)
        ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
        fig.tight_layout()

        combined_png_path = porewater_output_dir / f"Porewater_AllSeries_{loc}.png"
        fig.savefig(combined_png_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

        print(f"Saved combined pore-water dfs0 for {loc}: {combined_dfs0_path}")
        print(f"Saved combined pore-water plot for {loc}: {combined_png_path}")


In [ ]:
# TODO:

# Add GWL comparison
# For GWL Obs there is an excel file, I'm not sure exactly what is what

# Porewater Locations = C1, C2, C3
# Questions to answer:
# - Where are C1, C2, C3 located? Is it the locations in the shapefile?
# - what depth for lysimeters?
# - What depth for suction cups?

In [ ]:
# User-defined arguments

# Directories. Res_filepath points to the WQ_3DUZ.dfs3 file
Obs_dir = pathlib.Path(r"P:\WP1_PGM\ObservedData\Processed")
# Res_dir = pathlib.Path(r"C:\DHI\Cernici_060125\Cernici16_Ben_v101_WM_AD_v19_critDrought.she - Result Files")
# Res_filepath = Res_dir / "Cernici16_Ben_v101_WM_AD_v19_WQ_3DUZ.dfs3"
Res_dir = pathlib.Path(r"C:\DHI\Cernici_060125\Cernici16_Ben_v101_WM_AD_v20_v1_ALMM_JKLparams")
Res_filepath = Res_dir / "Cernici16_Ben_v101_WM_AD_v20_v1_ALMM_JKLparams_WQ_3DUZ.dfs3"

#Output directory default same as results
Output_dir = Res_filepath.parent / "ObsResCompare"

# Which obs do you want to investigate?
#       Options are: 'ET', 'Harvest_dry_grain\\BCc', 'LAI', 'Root Depth', 'Soil_water_N_P'
#       Note "GWL" comparison is not implemented, but observations exist
obs2compare = [
'ET',
'Harvest_dry_grain\\BCc',
'LAI',
'Root Depth',
'Soil_water_N_P'
]

# Dicts for comparing model results to the observations of each type
# dict entry = "your name key": ["Results state variable name", "Observations file name without extension"]
#       'Harvest_dry_grain\\BCc'
crops2compare = {'Spring Barley': ["BC1h","Spring barley_BCh",],
                 'Grass': ["BC2h","Permanent grassland_BCh"], # clover is a subset of grass -- will also need information on where it is to be able to handle it
                 'Winter Barley': ["BC3h","Winter barley_BCh"],
                 }

#       'ET'
ET2compare = {'Actual Evapotranspiration': ["EtA","ETa"],
              'Reference Evapotranspiration': ["RefEt","ETr"]
              }

#       'LAI' and 'Root Depth'
eo2compare = {'LAI': ["LAI","LAI_Mart_200_1981-2020"],
              'Root Depth': ["RD","Rd_Mart_200_1981-2020"]
              }

#'Soil_water_N_P'
porewater2compare = {'Porewater Column NH4': ["S_mNH4","Suction Cups"],
                     'Porewater Column NO3': ["S_mNO3","Suction Cups"],
                     'Groundwater Column NH4': ["S_mNH4","Lysimeter"],
                     'Groundwater Column NO3': ["S_mNO3","Lysimeter"],
                     'Porewater Layer NH4': ["NH4","Suction Cups"],
                     'Porewater Layer NO3': ["NO3","Suction Cups"],
                     'Groundwater Layer NH4': ["NH4","Lysimeter"],
                     'Groundwater Layer NO3': ["NO3","Lysimeter"]
                     }

# Extra user defined arguments for porewater
# Both sampling methods occured at 30-40cm depths, in MIKE SHE soil profile 3
# This means that the appropriate layer for them is layer 18, which is from 0.3-0.4m depth
lysimeter_layer = 18 # 0 # bottom
suctioncup_layer = 18 # 23 # top


In [ ]:
# Build required item list first, then read only those items from the model results file.
# This avoids loading the entire dfs3 into memory.

dfs_header = mikeio.open(Res_filepath)
all_item_names_raw = [item.name for item in dfs_header.items]
all_item_names = [name for name in all_item_names_raw if '(macropore phase)' not in name]

required_items = set()

# Harvest: uses substring lookup from crops2compare
if 'Harvest_dry_grain\\BCc' in obs2compare:
    for _, (res_name, _) in crops2compare.items():
        required_items.add(find_model_item_by_substring(res_name, all_item_names, Res_filepath.name))

# ET: uses substring lookup from ET2compare
if 'ET' in obs2compare:
    for _, (res_name, _) in ET2compare.items():
        required_items.add(find_model_item_by_substring(res_name, all_item_names, Res_filepath.name))

# LAI / Root Depth: processing code matches by suffix in model item names
if 'LAI' in obs2compare:
    lai_matches = [name for name in all_item_names if name.lower().rstrip().endswith(' lai')]
    if not lai_matches:
        raise ValueError(
            f"No model item ending with ' LAI' found in {Res_filepath.name}. Available items: {all_item_names}"
        )
    if len(lai_matches) > 1:
        raise ValueError(
            f"Multiple model items matched LAI suffix in {Res_filepath.name}: {lai_matches}"
        )
    required_items.add(lai_matches[0])

if 'Root Depth' in obs2compare:
    rd_matches = [name for name in all_item_names if name.lower().rstrip().endswith(' rd')]
    if not rd_matches:
        raise ValueError(
            f"No model item ending with ' RD' found in {Res_filepath.name}. Available items: {all_item_names}"
        )
    if len(rd_matches) > 1:
        raise ValueError(
            f"Multiple model items matched RD suffix in {Res_filepath.name}: {rd_matches}"
        )
    required_items.add(rd_matches[0])

# Porewater: match suffixes for required species + S_PWV
if 'Soil_water_N_P' in obs2compare:
    compare_model_items = sorted({mapping[0] for mapping in porewater2compare.values()})
    model_items_needed = sorted(set(compare_model_items + ['S_PWV']))
    for target in model_items_needed:
        required_items.add(match_model_item_name_by_suffix(target, all_item_names, Res_filepath.name))

required_items = sorted(required_items)
if not required_items:
    raise ValueError(
        "No required model items were selected from obs2compare. Check obs2compare and mapping dictionaries."
    )

print(f"Scanning model file: {Res_filepath}")
print(f"  Total items in file: {len(all_item_names_raw)}")
print(f"  Non-macropore candidate items: {len(all_item_names)}")
print(f"  Items needed by selected observation types: {len(required_items)}")
for name in required_items:
    print(f"    - {name}")

# Read only required items
res_ds = mikeio.read(Res_filepath, items=required_items)
res_item_names = [item.name for item in res_ds.items]
print(f"Loaded filtered model dataset with {len(res_item_names)} items.")

In [ ]:
# Clean the output directory before running comparisons
if Output_dir.exists():
    try:
        shutil.rmtree(Output_dir)
    except Exception as e:
        print(f"Warning: Could not remove {Output_dir}: {e}")
Output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# --- Execute comparison workflows based on user configuration ---
# This cell orchestrates all observation comparisons. Modify obs2compare in the user-defined arguments cell above to control which comparisons run.

print("=" * 80)
print("STARTING OBSERVATION COMPARISONS")
print("=" * 80)

# Harvest comparison
if 'Harvest_dry_grain\\BCc' in obs2compare:
    print("\n" + "=" * 80)
    print("HARVEST COMPARISON")
    print("=" * 80)
    run_harvest_comparison(res_ds, res_item_names, Res_filepath, Output_dir, Obs_dir, crops2compare)
else:
    print("\nSkipping Harvest comparison (not in obs2compare list)")

# Evapotranspiration comparison
if 'ET' in obs2compare:
    print("\n" + "=" * 80)
    print("EVAPOTRANSPIRATION COMPARISON")
    print("=" * 80)
    run_et_comparison(res_ds, res_item_names, Res_filepath, Output_dir, Obs_dir, ET2compare)
else:
    print("\nSkipping ET comparison (not in obs2compare list)")

# LAI and Root Depth comparison
if any(cat in obs2compare for cat in ['LAI', 'Root Depth']):
    print("\n" + "=" * 80)
    print("LAI AND ROOT DEPTH COMPARISON")
    print("=" * 80)
    run_lai_rd_comparison(res_ds, res_item_names, Res_filepath, Output_dir, Obs_dir, eo2compare, obs2compare)
else:
    print("\nSkipping LAI/RD comparison (not in obs2compare list)")

# Porewater comparison
if 'Soil_water_N_P' in obs2compare:
    print("\n" + "=" * 80)
    print("POREWATER N/P COMPARISON")
    print("=" * 80)
    run_porewater_comparison(res_ds, res_item_names, Res_filepath, Output_dir, Obs_dir, porewater2compare, lysimeter_layer, suctioncup_layer)
else:
    print("\nSkipping Porewater comparison (not in obs2compare list)")

print("\n" + "=" * 80)
print("COMPLETED ALL COMPARISONS")
print("=" * 80)
